# OpenKind — Phase 3B: Qwen3.5 Backbone Parity Reference

**Roadmap scope:** Phase 3.3, native Qwen3.5 backbone parity.

This notebook does **not** train, adapt, quantize, select, or modify the model. It treats the already-selected OpenKind profile as immutable and exports a precise reference for the Rust implementation.

Locked reference:

- OpenKind profile: `a047d6802c3f06f085b8`
- Base model: `Qwen/Qwen3.5-4B-Base`
- Revision: `1001bb4d826a52d1f399e183466143f4da7b741b`
- Rendering: `state_first`
- Rejection: `score_summary`
- Selected bundle SHA-256: `4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332`
- Public reference bundle: `cowWhySo/OpenKind-Qwen3.5-4B-StateFirst`

## What this notebook exports

1. Exact bundle and runtime identity.
2. Exact tokenizer/state-first segmentation checks against the existing golden fixtures.
3. Exact Qwen3.5 model configuration, layer types, module classes, parameter names/shapes/dtypes.
4. One **per-layer last-token trace** for a diagnostic candidate sequence.
5. Final 2,560-dimensional hidden vectors for every candidate in the nonfinal golden fixtures.
6. A minimal cached-continuation trace for one state → question → candidate path.
7. End-to-end reference probabilities from the existing selected head.
8. A Rust handoff contract and optional Rust-output comparator.

The output is intended to answer one question:

> Given the same finalized token IDs and selected model revision, where does a native Rust implementation first diverge from the Python/Transformers reference?

## 1. Run requirements

Use a fresh Colab GPU runtime. L4 is sufficient for this sequential FP32 parity export; A100 is also fine.

The notebook preserves the current Colab Torch build and pins the surrounding reference stack to the same versions used by the verified OpenKind export. It intentionally refuses a different Torch major/minor rather than silently changing numerical behavior.

In [1]:
import sys, subprocess, importlib.metadata as im, json, os
from pathlib import Path

def installed(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return None

torch_version = installed("torch")
if torch_version is None:
    raise RuntimeError("Select a Colab GPU runtime with PyTorch installed.")

if not torch_version.startswith("2.11."):
    raise RuntimeError(
        f"Reference requires Torch 2.11.x; current runtime has {torch_version}. "
        "Use the same Colab runtime family as Phase 2IJ / Phase 3A."
    )

constraint = Path("/content/openkind_phase3b_torch_constraint.txt")
constraint.write_text(f"torch=={torch_version}\n")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "-c", str(constraint),
    "transformers==5.17.0",
    "accelerate==1.14.0",
    "safetensors==0.8.0",
    "huggingface-hub",
    "numpy>=2,<3",
    "scipy>=1.16,<2",
    "psutil>=5,<8",
])

if installed("torch") != torch_version:
    raise RuntimeError("Torch changed unexpectedly during dependency installation.")

ENVIRONMENT = {
    "python": sys.version,
    "packages": {
        n: installed(n) for n in (
            "torch", "transformers", "accelerate", "safetensors",
            "numpy", "scipy", "huggingface-hub", "psutil"
        )
    },
}
print(json.dumps(ENVIRONMENT, indent=2))

{
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "packages": {
    "torch": "2.11.0+cu128",
    "transformers": "5.17.0",
    "accelerate": "1.14.0",
    "safetensors": "0.8.0",
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "huggingface-hub": "1.29.0",
    "psutil": "5.9.5"
  }
}


## 2. Mount Drive and authenticate Hugging Face

In [2]:
from google.colab import drive, userdata
drive.mount("/content/drive")

try:
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    print("No HF_TOKEN Colab secret found. Public model access may still work.")

Mounted at /content/drive


## 3. Resolve the immutable OpenKind reference

The notebook consumes the existing 2IJ selected bundle from Drive rather than recreating model-selection code. It also locates the latest completed Phase 3A result for lineage only; Phase 3A benchmark timings are not reused as backbone acceptance thresholds.

In [3]:
import hashlib, zipfile, shutil, time, platform
from datetime import datetime, timezone
from pathlib import Path

EXPECTED_PROFILE = "a047d6802c3f06f085b8"
EXPECTED_BUNDLE_SHA256 = "4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332"
EXPECTED_MODEL_ID = "Qwen/Qwen3.5-4B-Base"
EXPECTED_MODEL_REVISION = "1001bb4d826a52d1f399e183466143f4da7b741b"
HF_REFERENCE_REPO = "cowWhySo/OpenKind-Qwen3.5-4B-StateFirst"

COLAB_ROOT = Path("/content/drive/MyDrive/Colab Notebooks")
BUNDLE_ZIP = (
    COLAB_ROOT
    / "OpenKind_Phase2IJ_results"
    / "2ij_model_selection_screen_v2"
    / "model_bundle_2ij_model_selection_screen_v2.zip"
)
PHASE3A_ROOT = COLAB_ROOT / "OpenKind_Phase3A_results"

def sha256_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

if not BUNDLE_ZIP.is_file():
    raise FileNotFoundError(BUNDLE_ZIP)

bundle_sha = sha256_file(BUNDLE_ZIP)
if bundle_sha != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        f"Selected bundle changed: expected {EXPECTED_BUNDLE_SHA256}, got {bundle_sha}"
    )

def latest_completed_phase3a():
    if not PHASE3A_ROOT.is_dir():
        return None, None
    for d in sorted([p for p in PHASE3A_ROOT.iterdir() if p.is_dir()], reverse=True):
        p = d / "PHASE3A_SUMMARY.json"
        if not p.is_file():
            continue
        try:
            obj = json.loads(p.read_text())
        except Exception:
            continue
        if (
            obj.get("profile_id") == EXPECTED_PROFILE
            and obj.get("status") == "completed_notebook_scope"
        ):
            return d, obj
    return None, None

phase3a_dir, phase3a_summary = latest_completed_phase3a()

print("Bundle:", BUNDLE_ZIP)
print("Bundle SHA-256:", bundle_sha)
print("Phase 3A:", phase3a_dir)
if phase3a_summary:
    print("Phase 3A parity:", {
        "semantic": phase3a_summary.get("semantic_batched_parity_all_pass"),
        "high_k": phase3a_summary.get("high_k_parity_all_pass"),
        "repeatability_delta": phase3a_summary.get("repeatability_max_probability_delta"),
    })

Bundle: /content/drive/MyDrive/Colab Notebooks/OpenKind_Phase2IJ_results/2ij_model_selection_screen_v2/model_bundle_2ij_model_selection_screen_v2.zip
Bundle SHA-256: 4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332
Phase 3A: /content/drive/MyDrive/Colab Notebooks/OpenKind_Phase3A_results/20260920T024056Z
Phase 3A parity: {'semantic': True, 'high_k': True, 'repeatability_delta': 0.0}


## 4. Verify and unpack the selected model bundle

In [4]:
LOCAL_BUNDLE = Path("/content/openkind_phase3b/reference_bundle")
if LOCAL_BUNDLE.exists():
    shutil.rmtree(LOCAL_BUNDLE)
LOCAL_BUNDLE.mkdir(parents=True)

with zipfile.ZipFile(BUNDLE_ZIP) as z:
    z.extractall(LOCAL_BUNDLE)

manifest = json.loads((LOCAL_BUNDLE / "BUNDLE_MANIFEST.json").read_text())
profile = json.loads((LOCAL_BUNDLE / "profile.json").read_text())
protocol = json.loads((LOCAL_BUNDLE / "inference_protocol.json").read_text())
model_contract = json.loads((LOCAL_BUNDLE / "MODEL_CONTRACT.json").read_text())
golden = json.loads((LOCAL_BUNDLE / "golden.json").read_text())

if manifest["profile_id"] != EXPECTED_PROFILE:
    raise RuntimeError("Bundle profile identity mismatch.")
if manifest["base"]["id"] != EXPECTED_MODEL_ID:
    raise RuntimeError("Bundle base model mismatch.")
if manifest["base"]["revision"] != EXPECTED_MODEL_REVISION:
    raise RuntimeError("Bundle base revision mismatch.")
if profile.get("layout") != "state_first":
    raise RuntimeError("Backbone parity notebook requires the selected state-first renderer.")
if profile.get("kind") != "score_summary":
    raise RuntimeError("Unexpected selected rejection profile.")

for rel, expected in manifest["files"].items():
    path = LOCAL_BUNDLE / rel
    if not path.is_file():
        raise RuntimeError(f"Missing manifested bundle member: {rel}")
    got = sha256_file(path)
    if got != expected:
        raise RuntimeError(f"Bundle member hash mismatch: {rel}")

print("Verified bundle files:", len(manifest["files"]))
print("Golden requests:", len(golden))
print("Profile:", profile["profile_id"], profile["name"])

Verified bundle files: 31
Golden requests: 2
Profile: a047d6802c3f06f085b8 score_summary_seed17


## 5. Create a separate Phase 3B run

Outputs are written under:

`MyDrive/Colab Notebooks/OpenKind_Phase3B_BackboneParity_results/<run_id>/`

This phase is a parity/export study only.

In [5]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOCAL_RUN = Path("/content/openkind_phase3b") / RUN_ID
DRIVE_RUN = COLAB_ROOT / "OpenKind_Phase3B_BackboneParity_results" / RUN_ID
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
DRIVE_RUN.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "schema": "openkind-phase3b-backbone-parity-config/v1",
    "run_id": RUN_ID,
    "profile_id": EXPECTED_PROFILE,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "base_model": EXPECTED_MODEL_ID,
    "base_revision": EXPECTED_MODEL_REVISION,
    "hf_reference_repo": HF_REFERENCE_REPO,
    "training_performed": False,
    "model_selection_performed": False,
    "model_modified": False,
    "reference_dtype": "float32",
    "reference_arithmetic": "strict_fp32_math_attention",
    "fixture_source": "selected bundle golden.json / policy_dev nonfinal fixtures",
    "final_labels_exported": False,
}
(LOCAL_RUN / "config.json").write_text(json.dumps(CONFIG, indent=2))
(DRIVE_RUN / "config.json").write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))
print("Drive run:", DRIVE_RUN)

{
  "schema": "openkind-phase3b-backbone-parity-config/v1",
  "run_id": "20260920T152206Z",
  "profile_id": "a047d6802c3f06f085b8",
  "bundle_sha256": "4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332",
  "base_model": "Qwen/Qwen3.5-4B-Base",
  "base_revision": "1001bb4d826a52d1f399e183466143f4da7b741b",
  "hf_reference_repo": "cowWhySo/OpenKind-Qwen3.5-4B-StateFirst",
  "training_performed": false,
  "model_selection_performed": false,
  "model_modified": false,
  "reference_dtype": "float32",
  "reference_arithmetic": "strict_fp32_math_attention",
  "fixture_source": "selected bundle golden.json / policy_dev nonfinal fixtures",
  "final_labels_exported": false
}
Drive run: /content/drive/MyDrive/Colab Notebooks/OpenKind_Phase3B_BackboneParity_results/20260920T152206Z


## 6. Import the exact bundled reference implementation

The implementation snapshot stored in the selected bundle is used only to reproduce the exact renderer/runtime contract. It is preferable here to importing a newer workbench from an unrelated checkout.

In [6]:
import sys, copy, gc
IMPL = LOCAL_BUNDLE / "implementation"
sys.path.insert(0, str(IMPL))

from odij_runtime import Runtime, segments, configure_arithmetic, walk_tensors, cache_bytes, tensor_digest
from odij_serve import request_rows
from odij_bundle import BundlePredictor
from odij_core import digest

spec = copy.deepcopy(profile["spec"])
spec["local_tokenizer_path"] = str(LOCAL_BUNDLE / "tokenizer")

print("Bundled runtime implementation:", IMPL)
print("Renderer layout:", profile["layout"])
print("Protocol max_tokens:", protocol["max_tokens"])

Bundled runtime implementation: /content/openkind_phase3b/reference_bundle/implementation
Renderer layout: state_first
Protocol max_tokens: 1792


## 7. Load the exact frozen Qwen3.5 text backbone

In [7]:
import torch, numpy as np, psutil

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required; no accidental CPU full-backbone fallback.")

gpu_name = torch.cuda.get_device_name(0)
free_b, total_b = torch.cuda.mem_get_info()

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

rt = Runtime(
    spec,
    LOCAL_RUN / "backbone_runtime",
    protocol,
    training=False,
    allow_cpu_test=False,
)

if type(rt.model).__name__ != "Qwen3_5TextModel":
    raise RuntimeError(f"Unexpected model class: {type(rt.model).__name__}")
if rt.hidden != 2560:
    raise RuntimeError(f"Unexpected hidden width: {rt.hidden}")
if sum(p.numel() for p in rt.model.parameters()) != 4205751296:
    raise RuntimeError("Pinned Qwen3.5-4B text parameter count changed.")

runtime_identity = {
    "gpu": gpu_name,
    "gpu_total_gib": total_b / 2**30,
    "gpu_free_gib_before_load": free_b / 2**30,
    "environment": ENVIRONMENT,
    "profile_id": EXPECTED_PROFILE,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "runtime_identity": rt.identity,
    "runtime_report": rt.report,
}
for path in [LOCAL_RUN / "runtime_identity.json", DRIVE_RUN / "runtime_identity.json"]:
    path.write_text(json.dumps(runtime_identity, indent=2, default=str))

print("Loaded:", type(rt.model).__name__)
print("GPU:", gpu_name)
print("Parameter count:", rt.report["parameter_count"])
print("After-load memory:", rt.report["after_load_memory"])

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Loaded: Qwen3_5TextModel
GPU: NVIDIA A100-SXM4-80GB
Parameter count: 4205751296
After-load memory: {'rss_mib': 1833.40234375, 'process_peak_rss_mib': 10048.6875, 'allocated_gib': 15.667662620544434, 'reserved_gib': 15.6875, 'peak_allocated_gib': 15.66796875, 'free_gib': 63.07110595703125, 'total_gib': 79.250732421875}


## 8. Export the architecture and tensor-name manifest

Rust should not infer the architecture from a marketing model name. This cell records the exact config loaded by the verified Python reference plus every model parameter name, shape, and dtype.

In [8]:
def module_name_map(model):
    return {name: type(module).__name__ for name, module in model.named_modules()}

param_inventory = []
for name, tensor in rt.model.state_dict().items():
    param_inventory.append({
        "name": name,
        "shape": list(tensor.shape),
        "dtype": str(tensor.dtype).replace("torch.", ""),
        "numel": int(tensor.numel()),
    })

layer_manifest = []
for i, layer in enumerate(rt.model.layers):
    layer_manifest.append({
        "index": i,
        "declared_type": rt.model.config.layer_types[i],
        "module_class": type(layer).__name__,
        "token_mixer_class": (
            type(layer.linear_attn).__name__
            if getattr(layer, "block_type", None) == "linear_attention"
            else type(layer.self_attn).__name__
        ),
        "mlp_class": type(layer.mlp).__name__,
    })

architecture = {
    "schema": "openkind-qwen35-backbone-architecture/v1",
    "profile_id": EXPECTED_PROFILE,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "base_model": EXPECTED_MODEL_ID,
    "base_revision": EXPECTED_MODEL_REVISION,
    "parameter_count": int(sum(p.numel() for p in rt.model.parameters())),
    "model_class": type(rt.model).__name__,
    "config": rt.model.config.to_dict(),
    "layers": layer_manifest,
    "parameter_inventory": param_inventory,
}

for path in [LOCAL_RUN / "QWEN35_ARCHITECTURE.json", DRIVE_RUN / "QWEN35_ARCHITECTURE.json"]:
    path.write_text(json.dumps(architecture, indent=2, default=str))

print("Layers:", len(layer_manifest))
print("Layer types:", {x["declared_type"] for x in layer_manifest})
print("Parameter tensors:", len(param_inventory))
print("Hidden:", rt.model.config.hidden_size)

Layers: 32
Layer types: {'full_attention', 'linear_attention'}
Parameter tensors: 426
Hidden: 2560


## 9. Reconstruct and verify exact state-first tokenization

The existing bundle already contains token fixtures. This cell regenerates them from the bundled tokenizer and bundled renderer and requires exact integer equality.

This is the hard boundary for Rust tokenizer/renderer parity: **no fuzzy text equivalence**.

In [9]:
token_fixture_records = []
all_rows = []

for case_idx, item in enumerate(golden):
    rows = request_rows(item["request"], protocol)
    saved = {x["question_id"]: x for x in item["head_and_token_fixtures"]}

    for row in rows:
        qid = row["q"]["id"]
        expected = saved[qid]
        root_ids, question_ids, candidate_suffix_ids, full_candidate_ids = segments(
            row, rt.tokenizer, profile["layout"], protocol["max_tokens"]
        )

        checks = {
            "root_ids_exact": root_ids == expected["root_ids"],
            "question_ids_exact": question_ids == expected["question_ids"],
            "candidate_suffix_ids_exact": candidate_suffix_ids == expected["candidate_suffix_ids"],
            "full_candidate_ids_exact": full_candidate_ids == expected["full_candidate_ids"],
        }
        if not all(checks.values()):
            raise RuntimeError(f"Token fixture mismatch for {qid}: {checks}")

        rec = {
            "fixture_case": case_idx,
            "question_id": qid,
            "root_ids": root_ids,
            "question_ids": question_ids,
            "candidate_suffix_ids": candidate_suffix_ids,
            "full_candidate_ids": full_candidate_ids,
            "checks": checks,
        }
        token_fixture_records.append(rec)
        all_rows.append((case_idx, row, expected, rec))

token_fixture_export = {
    "schema": "openkind-qwen35-token-fixtures/v1",
    "profile_id": EXPECTED_PROFILE,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "tokenizer_backend_hash": rt.identity["tokenizer_backend"],
    "layout": profile["layout"],
    "segmentation": model_contract["feature_contract"]["causal_tokenization"],
    "records": token_fixture_records,
}
for path in [LOCAL_RUN / "TOKEN_FIXTURES.json", DRIVE_RUN / "TOKEN_FIXTURES.json"]:
    path.write_text(json.dumps(token_fixture_export, indent=2))

print("Exact token fixtures passed:", len(token_fixture_records))
print("Candidate sequences:", sum(len(x["full_candidate_ids"]) for x in token_fixture_records))

Exact token fixtures passed: 4
Candidate sequences: 10


## 10. Helpers for tensor fingerprints and Rust-friendly fixture names

The Safetensors file carries the actual float32 reference vectors. JSON carries identity, shapes, hashes and diagnostics.

In [10]:
from safetensors.torch import save_file, load_file

def tensor_sha256(t):
    arr = t.detach().contiguous().cpu()
    return hashlib.sha256(arr.numpy().tobytes()).hexdigest()

def tensor_stats(t):
    x = t.detach().float().cpu()
    return {
        "shape": list(x.shape),
        "dtype": "float32",
        "sha256": tensor_sha256(x),
        "min": float(x.min()),
        "max": float(x.max()),
        "mean": float(x.mean()),
        "l2": float(torch.linalg.vector_norm(x)),
    }

def safe_key(text):
    return "".join(c if c.isalnum() or c in "._-" else "_" for c in str(text))

reference_tensors = {}
reference_index = {}

## 11. Full-sequence backbone reference for every golden candidate

This is the primary **Phase 3.3** parity surface.

For every candidate sequence already present in the nonfinal bundle fixtures:

1. use the exact saved token IDs;
2. run the frozen Qwen text backbone in strict FP32;
3. save the final last-token 2,560-D hidden vector;
4. compare it with the older candidate feature already exported in `golden.json`.

The older feature comparison protects this notebook from accidentally redefining what OpenKind calls the candidate feature.

In [11]:
full_sequence_records = []
max_delta_vs_existing_golden = 0.0

for case_idx, row, expected, tokrec in all_rows:
    qid = row["q"]["id"]
    saved_features = np.asarray(expected["candidate_features"], dtype=np.float32)

    if len(saved_features) != len(tokrec["full_candidate_ids"]):
        raise RuntimeError(f"Golden candidate-feature count mismatch for {qid}")

    for cand_idx, ids in enumerate(tokrec["full_candidate_ids"]):
        with torch.no_grad():
            out = rt.model(
                **rt.batch(ids),
                use_cache=False,
                return_dict=True,
            )
        vec = out.last_hidden_state[0, -1, :].float().detach().cpu()
        expected_vec = torch.from_numpy(saved_features[cand_idx])
        delta = float(torch.max(torch.abs(vec - expected_vec)))
        max_delta_vs_existing_golden = max(max_delta_vs_existing_golden, delta)

        key = f"fixture_{case_idx}.{safe_key(qid)}.candidate_{cand_idx}.final_hidden"
        reference_tensors[key] = vec
        reference_index[key] = tensor_stats(vec)

        full_sequence_records.append({
            "fixture_case": case_idx,
            "question_id": qid,
            "candidate_index": cand_idx,
            "token_count": len(ids),
            "tensor_key": key,
            "max_abs_delta_vs_existing_bundle_feature": delta,
        })

print("Saved full-sequence candidate vectors:", len(full_sequence_records))
print("Max delta vs existing bundle candidate_features:", max_delta_vs_existing_golden)

# This should be tiny because both use the same frozen runtime contract.
if max_delta_vs_existing_golden > 1e-4:
    raise RuntimeError(
        "Fresh backbone reference diverged materially from the selected bundle's existing candidate features."
    )

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Saved full-sequence candidate vectors: 10
Max delta vs existing bundle candidate_features: 4.9591064453125e-05


## 12. Per-layer localization trace

One deterministic candidate sequence is traced through:

- input embedding output,
- all 32 decoder blocks,
- final RMSNorm.

Only the **last token vector** is retained at each stage, so the trace is small enough to use routinely from Rust tests.

If Rust's final hidden vector differs, compare stage-by-stage to find the first divergent layer.

In [12]:
diagnostic = all_rows[0]
case_idx, row, expected, tokrec = diagnostic
diag_ids = tokrec["full_candidate_ids"][0]
diag_qid = row["q"]["id"]

captures = {}
hooks = []

def capture_hook(name):
    def hook(module, args, output):
        value = output[0] if isinstance(output, (tuple, list)) else output
        if not isinstance(value, torch.Tensor):
            raise RuntimeError(f"Hook {name} did not return a tensor")
        captures[name] = value[0, -1, :].detach().float().cpu()
    return hook

hooks.append(rt.model.embed_tokens.register_forward_hook(capture_hook("embedding")))
for i, layer in enumerate(rt.model.layers):
    hooks.append(layer.register_forward_hook(capture_hook(f"layer_{i:02d}")))
hooks.append(rt.model.norm.register_forward_hook(capture_hook("final_norm")))

try:
    with torch.no_grad():
        diag_out = rt.model(
            **rt.batch(diag_ids),
            use_cache=False,
            return_dict=True,
        )
finally:
    for h in hooks:
        h.remove()

expected_names = ["embedding"] + [f"layer_{i:02d}" for i in range(32)] + ["final_norm"]
missing = [x for x in expected_names if x not in captures]
if missing:
    raise RuntimeError(f"Missing diagnostic hook captures: {missing}")

final_direct = diag_out.last_hidden_state[0, -1, :].detach().float().cpu()
final_norm_delta = float(torch.max(torch.abs(final_direct - captures["final_norm"])))
if final_norm_delta > 1e-6:
    raise RuntimeError(f"Final norm hook differs from last_hidden_state: {final_norm_delta}")

trace_index = []
for stage in expected_names:
    key = f"diagnostic.{stage}"
    reference_tensors[key] = captures[stage]
    reference_index[key] = tensor_stats(captures[stage])
    trace_index.append({
        "stage": stage,
        "tensor_key": key,
        **reference_index[key],
    })

diagnostic_trace = {
    "schema": "openkind-qwen35-layer-trace/v1",
    "profile_id": EXPECTED_PROFILE,
    "fixture_case": case_idx,
    "question_id": diag_qid,
    "candidate_index": 0,
    "input_ids": diag_ids,
    "token_count": len(diag_ids),
    "stages": trace_index,
    "final_norm_delta_vs_model_output": final_norm_delta,
}

for path in [LOCAL_RUN / "LAYER_TRACE.json", DRIVE_RUN / "LAYER_TRACE.json"]:
    path.write_text(json.dumps(diagnostic_trace, indent=2))

print("Diagnostic sequence:", diag_qid, "tokens:", len(diag_ids))
print("Trace stages:", len(trace_index))

Diagnostic sequence: asset tokens: 121
Trace stages: 34


## 13. One cached state → question → candidate continuation trace

Full-sequence parity comes first. This additional trace helps the next step by pinning one continuation path using the same hybrid cache semantics validated in Phase 3A.

It does **not** replace the Phase 3A `BranchableState` fixture set.

In [13]:
root_ids = tokrec["root_ids"]
question_ids = tokrec["question_ids"]
candidate_ids = tokrec["candidate_suffix_ids"][0]

with torch.no_grad():
    root_out = rt.model(
        **rt.batch(root_ids),
        use_cache=True,
        return_dict=True,
    )
root_cache = root_out.past_key_values
root_hidden = root_out.last_hidden_state[0, -1, :].detach().float().cpu()

with torch.no_grad():
    question_out = rt.model(
        **rt.batch(question_ids, prefix=len(root_ids)),
        past_key_values=copy.deepcopy(root_cache),
        use_cache=True,
        return_dict=True,
    )
question_cache = question_out.past_key_values
question_hidden = question_out.last_hidden_state[0, -1, :].detach().float().cpu()

with torch.no_grad():
    candidate_out = rt.model(
        **rt.batch(candidate_ids, prefix=len(root_ids) + len(question_ids)),
        past_key_values=copy.deepcopy(question_cache),
        use_cache=True,
        return_dict=True,
    )
candidate_hidden = candidate_out.last_hidden_state[0, -1, :].detach().float().cpu()

full_key = f"fixture_{case_idx}.{safe_key(diag_qid)}.candidate_0.final_hidden"
cached_vs_full = float(torch.max(torch.abs(candidate_hidden - reference_tensors[full_key])))

for key, tensor in {
    "continuation.root_last_hidden": root_hidden,
    "continuation.question_last_hidden": question_hidden,
    "continuation.candidate_last_hidden": candidate_hidden,
}.items():
    reference_tensors[key] = tensor
    reference_index[key] = tensor_stats(tensor)

continuation = {
    "schema": "openkind-qwen35-continuation-trace/v1",
    "profile_id": EXPECTED_PROFILE,
    "question_id": diag_qid,
    "root_ids": root_ids,
    "question_ids": question_ids,
    "candidate_suffix_ids": candidate_ids,
    "root_cache_bytes": int(cache_bytes(root_cache)),
    "root_cache_digest": tensor_digest(root_cache),
    "root_position": int(root_cache.get_seq_length()),
    "question_position": int(question_cache.get_seq_length()),
    "candidate_position": int(candidate_out.past_key_values.get_seq_length()),
    "cached_candidate_max_abs_delta_vs_full_sequence": cached_vs_full,
}

if cached_vs_full > 1e-4:
    raise RuntimeError(f"Cached continuation differs from full sequence: {cached_vs_full}")

for path in [LOCAL_RUN / "CONTINUATION_TRACE.json", DRIVE_RUN / "CONTINUATION_TRACE.json"]:
    path.write_text(json.dumps(continuation, indent=2))

print(json.dumps(continuation, indent=2))

{
  "schema": "openkind-qwen35-continuation-trace/v1",
  "profile_id": "a047d6802c3f06f085b8",
  "question_id": "asset",
  "root_ids": [
    79503,
    279,
    16713,
    5307,
    12521,
    13,
    5272,
    279,
    1528,
    430,
    5721,
    11,
    524,
    430,
    10895,
    13,
    198,
    1349,
    25,
    198,
    37,
    2399,
    278,
    10142,
    264,
    18,
    22084,
    24,
    22,
    17,
    895,
    24,
    67,
    21,
    15,
    69,
    19,
    13,
    561,
    17695,
    6473,
    513,
    67765,
    13,
    9457,
    1514,
    9052,
    25,
    3638,
    3421,
    13,
    25009,
    1928,
    369,
    7167,
    13,
    12932,
    5105,
    557,
    524,
    10457,
    13,
    561,
    9052,
    10042,
    1785,
    369,
    13928,
    13,
    561,
    25530,
    5253,
    2119,
    369,
    220,
    17,
    13,
    1473,
    8976,
    8833,
    557,
    524,
    12784,
    13,
    4081,
    11889,
    513,
    279,
    2400,
    5721,
    3150,
    26,
   

## 14. End-to-end selected-profile probability reference

Backbone vectors are diagnostic. The actual same-profile implementation gate ultimately lives at the decision distribution and policy boundary.

This cell reloads the already-selected bundle predictor and evaluates the same nonfinal requests. No labels or final data are used.

In [14]:
bundle_predictor = BundlePredictor(
    LOCAL_BUNDLE,
    LOCAL_RUN / "bundle_probability_runtime",
    allow_cpu_test=False,
)

probability_reference = []
try:
    for case_idx, item in enumerate(golden):
        result = bundle_predictor.evaluate(item["request"])
        probability_reference.append({
            "fixture_case": case_idx,
            "request_id": result["request_id"],
            "engine_profile": result["engine_profile"],
            "answers": result["answers"],
        })
finally:
    bundle_predictor.close()

prob_export = {
    "schema": "openkind-qwen35-probability-reference/v1",
    "profile_id": EXPECTED_PROFILE,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "probability_tolerance": protocol["probability_tolerance"],
    "order_tolerance": protocol["order_tolerance"],
    "require_zero_argmax_changes": True,
    "final_labels_exported": False,
    "records": probability_reference,
}
for path in [LOCAL_RUN / "PROBABILITY_REFERENCE.json", DRIVE_RUN / "PROBABILITY_REFERENCE.json"]:
    path.write_text(json.dumps(prob_export, indent=2))

print("Reference requests:", len(probability_reference))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Reference requests: 2


## 15. Save the Rust-facing Safetensors reference

In [15]:
REFERENCE_TENSORS = LOCAL_RUN / "QWEN35_BACKBONE_GOLDEN.safetensors"
save_file(reference_tensors, str(REFERENCE_TENSORS))

tensor_index_export = {
    "schema": "openkind-qwen35-backbone-tensor-index/v1",
    "profile_id": EXPECTED_PROFILE,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "safetensors_sha256": sha256_file(REFERENCE_TENSORS),
    "tensors": reference_index,
    "full_sequence_records": full_sequence_records,
}

for path in [LOCAL_RUN / "BACKBONE_TENSOR_INDEX.json", DRIVE_RUN / "BACKBONE_TENSOR_INDEX.json"]:
    path.write_text(json.dumps(tensor_index_export, indent=2))

shutil.copy2(REFERENCE_TENSORS, DRIVE_RUN / REFERENCE_TENSORS.name)

print("Reference tensors:", len(reference_tensors))
print("Safetensors SHA-256:", tensor_index_export["safetensors_sha256"])
print("Size MiB:", REFERENCE_TENSORS.stat().st_size / 2**20)

Reference tensors: 47
Safetensors SHA-256: b879a03a42f9b62b440bd6ff1a0b226e7690c50bbcc63689f99b62a704a8f90f
Size MiB: 0.46300506591796875


## 16. Backbone parity contract for Rust

Hidden-vector differences are useful localization diagnostics, but this notebook does **not invent a new hidden-state tolerance** for Metal/Candle.

The invariant hierarchy is:

1. model/profile/revision/config identity — exact;
2. tokenizer and finalized token IDs — exact;
3. tensor shapes and stage ordering — exact;
4. hidden-state deltas — measured and reported for localization;
5. selected-profile probabilities — historical same-profile tolerance;
6. argmax/policy outputs — zero changes under the declared same-profile gate.

This prevents a convenient hidden-state threshold from silently redefining the model contract.

In [16]:
BACKBONE_CONTRACT = {
    "schema": "openkind-qwen35-backbone-parity-contract/v1",
    "profile_id": EXPECTED_PROFILE,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "hf_reference_repo": HF_REFERENCE_REPO,
    "base": {
        "model_id": EXPECTED_MODEL_ID,
        "revision": EXPECTED_MODEL_REVISION,
        "parameter_count": 4205751296,
        "hidden_size": 2560,
        "dtype": "float32",
        "transformers": "5.17.0",
        "torch_family": "2.11.x",
    },
    "exact_gates": {
        "profile_id": True,
        "base_revision": True,
        "renderer_layout": "state_first",
        "token_ids": True,
        "tensor_shapes": True,
        "layer_order": True,
    },
    "diagnostic_numeric_checks": {
        "hidden_vectors": "report max_abs, RMS, cosine and first divergent layer; no arbitrary Metal threshold declared here",
        "layer_trace": "embedding -> 32 decoder layers -> final RMSNorm",
    },
    "decision_gate": {
        "max_probability_delta": float(protocol["probability_tolerance"]),
        "order_tolerance": float(protocol["order_tolerance"]),
        "require_zero_argmax_changes": True,
        "require_zero_policy_changes": True,
    },
    "execution_reference": {
        "full_sequence_first": True,
        "cached_continuation_second": True,
        "branchable_state_after_full_backbone_parity": True,
        "no_generation": True,
    },
    "artifacts": {
        "tokens": "TOKEN_FIXTURES.json",
        "architecture": "QWEN35_ARCHITECTURE.json",
        "layer_trace": "LAYER_TRACE.json",
        "continuation_trace": "CONTINUATION_TRACE.json",
        "tensor_index": "BACKBONE_TENSOR_INDEX.json",
        "tensor_values": "QWEN35_BACKBONE_GOLDEN.safetensors",
        "probabilities": "PROBABILITY_REFERENCE.json",
    },
}

for path in [LOCAL_RUN / "BACKBONE_PARITY_CONTRACT.json", DRIVE_RUN / "BACKBONE_PARITY_CONTRACT.json"]:
    path.write_text(json.dumps(BACKBONE_CONTRACT, indent=2))

print(json.dumps(BACKBONE_CONTRACT, indent=2))

{
  "schema": "openkind-qwen35-backbone-parity-contract/v1",
  "profile_id": "a047d6802c3f06f085b8",
  "bundle_sha256": "4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332",
  "hf_reference_repo": "cowWhySo/OpenKind-Qwen3.5-4B-StateFirst",
  "base": {
    "model_id": "Qwen/Qwen3.5-4B-Base",
    "revision": "1001bb4d826a52d1f399e183466143f4da7b741b",
    "parameter_count": 4205751296,
    "hidden_size": 2560,
    "dtype": "float32",
    "transformers": "5.17.0",
    "torch_family": "2.11.x"
  },
  "exact_gates": {
    "profile_id": true,
    "base_revision": true,
    "renderer_layout": "state_first",
    "token_ids": true,
    "tensor_shapes": true,
    "layer_order": true
  },
  "diagnostic_numeric_checks": {
    "hidden_vectors": "report max_abs, RMS, cosine and first divergent layer; no arbitrary Metal threshold declared here",
    "layer_trace": "embedding -> 32 decoder layers -> final RMSNorm"
  },
  "decision_gate": {
    "max_probability_delta": 0.005,
    "order_t

## 17. Optional comparator for Rust output

### Expected Rust interchange

For easiest early integration, have the Rust probe write either:

- `RUST_BACKBONE_OUTPUTS.safetensors` with keys matching `QWEN35_BACKBONE_GOLDEN.safetensors`, or
- `RUST_BACKBONE_OUTPUTS.json` mapping tensor keys to float arrays.

Copy the file into this run's Drive folder and rerun this cell.

The comparator reports hidden deltas; it does not reinterpret them as a new acceptance threshold.

In [17]:
RUST_SAFE = DRIVE_RUN / "RUST_BACKBONE_OUTPUTS.safetensors"
RUST_JSON = DRIVE_RUN / "RUST_BACKBONE_OUTPUTS.json"

rust_tensors = None
source = None

if RUST_SAFE.is_file():
    rust_tensors = {k: v.float().cpu() for k, v in load_file(str(RUST_SAFE)).items()}
    source = RUST_SAFE.name
elif RUST_JSON.is_file():
    obj = json.loads(RUST_JSON.read_text())
    rust_tensors = {k: torch.tensor(v, dtype=torch.float32) for k, v in obj.items()}
    source = RUST_JSON.name

if rust_tensors is None:
    print("No Rust output file found yet. This is expected while the Rust backbone is still being built.")
else:
    comparisons = []
    for key, ref in reference_tensors.items():
        if key not in rust_tensors:
            comparisons.append({"key": key, "status": "missing_in_rust"})
            continue
        got = rust_tensors[key].reshape(ref.shape).float()
        diff = got - ref.float()
        denom = float(torch.linalg.vector_norm(ref.float()))
        cosine = float(torch.nn.functional.cosine_similarity(
            got.reshape(1, -1), ref.float().reshape(1, -1)
        ))
        comparisons.append({
            "key": key,
            "status": "compared",
            "shape": list(ref.shape),
            "max_abs": float(torch.max(torch.abs(diff))),
            "rms": float(torch.sqrt(torch.mean(diff * diff))),
            "relative_l2": float(torch.linalg.vector_norm(diff)) / max(denom, 1e-30),
            "cosine": cosine,
        })

    result = {
        "schema": "openkind-rust-backbone-comparison/v1",
        "profile_id": EXPECTED_PROFILE,
        "rust_source": source,
        "comparisons": comparisons,
    }
    for path in [LOCAL_RUN / "RUST_BACKBONE_COMPARISON.json", DRIVE_RUN / "RUST_BACKBONE_COMPARISON.json"]:
        path.write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2)[:20000])

No Rust output file found yet. This is expected while the Rust backbone is still being built.


## 18. Create the Rust handoff

The recommended native implementation order after this notebook is:

1. exact tokenizer/state-first segmentation;
2. embedding lookup;
3. Qwen3.5 decoder forward in full-sequence FP32;
4. stage-by-stage comparison against `LAYER_TRACE.json`;
5. final candidate hidden-vector comparison;
6. selected head/probability comparison;
7. only then cached continuation and `BranchableState`.

In [18]:
handoff = f"""# OpenKind Phase 3B — Qwen3.5 Rust backbone handoff

Profile: `{EXPECTED_PROFILE}`
Bundle SHA-256: `{EXPECTED_BUNDLE_SHA256}`
Base: `{EXPECTED_MODEL_ID}`
Revision: `{EXPECTED_MODEL_REVISION}`
Reference repo: `{HF_REFERENCE_REPO}`

Training/model selection/model modification in Phase 3B: **none**

## Rust implementation order

1. Load the exact model revision and exported tokenizer.
2. Reproduce `TOKEN_FIXTURES.json` exactly.
3. Reproduce Qwen3.5 full-sequence FP32 inference.
4. Use `LAYER_TRACE.json` + `QWEN35_BACKBONE_GOLDEN.safetensors` to identify the first divergent stage.
5. Reproduce every golden candidate's final 2,560-D hidden vector closely enough that the selected decision distribution satisfies the existing same-profile gate.
6. Reproduce `PROBABILITY_REFERENCE.json` within probability tolerance `{protocol["probability_tolerance"]}` with zero argmax/policy changes.
7. Reproduce `CONTINUATION_TRACE.json`.
8. Then implement the Phase 3A `BranchableState` contract and batched-Q/K execution.

## Important Qwen3.5 requirements

The text backbone is hybrid. The exact model config in `QWEN35_ARCHITECTURE.json` is authoritative.
Do not implement it as a conventional KV-only transformer.

The selected 4B config has 32 decoder blocks with the declared `layer_types` exported by the notebook.
Continuation state must preserve full-attention KV plus linear-attention recurrent and convolution state.

## Do not silently change

- renderer or segment tokenization;
- model revision;
- normalization/head/rejection parameters;
- dtype/precision while claiming strict reference parity;
- option ordering;
- candidate descriptions;
- profile identity.

CUDA timing from this notebook is not a Mac/Metal acceptance threshold.
"""

for path in [LOCAL_RUN / "RUST_BACKBONE_HANDOFF.md", DRIVE_RUN / "RUST_BACKBONE_HANDOFF.md"]:
    path.write_text(handoff)

print(handoff)

# OpenKind Phase 3B — Qwen3.5 Rust backbone handoff

Profile: `a047d6802c3f06f085b8`
Bundle SHA-256: `4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332`
Base: `Qwen/Qwen3.5-4B-Base`
Revision: `1001bb4d826a52d1f399e183466143f4da7b741b`
Reference repo: `cowWhySo/OpenKind-Qwen3.5-4B-StateFirst`

Training/model selection/model modification in Phase 3B: **none**

## Rust implementation order

1. Load the exact model revision and exported tokenizer.
2. Reproduce `TOKEN_FIXTURES.json` exactly.
3. Reproduce Qwen3.5 full-sequence FP32 inference.
4. Use `LAYER_TRACE.json` + `QWEN35_BACKBONE_GOLDEN.safetensors` to identify the first divergent stage.
5. Reproduce every golden candidate's final 2,560-D hidden vector closely enough that the selected decision distribution satisfies the existing same-profile gate.
6. Reproduce `PROBABILITY_REFERENCE.json` within probability tolerance `0.005` with zero argmax/policy changes.
7. Reproduce `CONTINUATION_TRACE.json`.
8. Then implement the P

## 19. Finish and archive the Phase 3B reference

In [19]:
# Close the resident backbone before packaging.
try:
    rt.close()
except Exception as e:
    print("Runtime close warning:", e)

summary = {
    "schema": "openkind-phase3b-backbone-parity-summary/v1",
    "status": "completed_notebook_scope",
    "run_id": RUN_ID,
    "profile_id": EXPECTED_PROFILE,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "base_model": EXPECTED_MODEL_ID,
    "base_revision": EXPECTED_MODEL_REVISION,
    "training_performed": False,
    "model_selection_performed": False,
    "model_modified": False,
    "token_fixture_records": len(token_fixture_records),
    "full_sequence_candidate_vectors": len(full_sequence_records),
    "diagnostic_trace_stages": len(trace_index),
    "max_delta_vs_existing_bundle_feature": max_delta_vs_existing_golden,
    "cached_candidate_max_abs_delta_vs_full_sequence": cached_vs_full,
    "reference_tensor_count": len(reference_tensors),
    "rust_parity_claim": False,
    "metal_parity_claim": False,
    "next_step": "Run Rust full-sequence backbone probe against these fixtures; localize first divergent layer before BranchableState optimization.",
}

for path in [LOCAL_RUN / "PHASE3B_SUMMARY.json", DRIVE_RUN / "PHASE3B_SUMMARY.json"]:
    path.write_text(json.dumps(summary, indent=2))

archive_path = shutil.make_archive(
    str(LOCAL_RUN / f"OpenKind_Phase3B_BackboneParity_{RUN_ID}"),
    "zip",
    root_dir=LOCAL_RUN,
)
shutil.copy2(archive_path, DRIVE_RUN / Path(archive_path).name)

print(json.dumps(summary, indent=2))
print("Drive output:", DRIVE_RUN)
print("Archive:", DRIVE_RUN / Path(archive_path).name)

{
  "schema": "openkind-phase3b-backbone-parity-summary/v1",
  "status": "completed_notebook_scope",
  "run_id": "20260920T152206Z",
  "profile_id": "a047d6802c3f06f085b8",
  "bundle_sha256": "4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332",
  "base_model": "Qwen/Qwen3.5-4B-Base",
  "base_revision": "1001bb4d826a52d1f399e183466143f4da7b741b",
  "training_performed": false,
  "model_selection_performed": false,
  "model_modified": false,
  "token_fixture_records": 4,
  "full_sequence_candidate_vectors": 10,
  "diagnostic_trace_stages": 34,
  "max_delta_vs_existing_bundle_feature": 4.9591064453125e-05,
  "cached_candidate_max_abs_delta_vs_full_sequence": 1.9073486328125e-05,
  "reference_tensor_count": 47,
  "rust_parity_claim": false,
  "metal_parity_claim": false,
  "next_step": "Run Rust full-sequence backbone probe against these fixtures; localize first divergent layer before BranchableState optimization."
}
Drive output: /content/drive/MyDrive/Colab Notebooks/OpenK

## 20. Cleanup

Use this only after the export finishes and all output files are visible in Drive.

In [20]:
from google.colab import runtime
print("Phase 3B artifacts are saved. Disconnect manually when ready, or uncomment:")
# runtime.unassign()

Phase 3B artifacts are saved. Disconnect manually when ready, or uncomment:
